# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
import xgboost as xgb
from xgboost import XGBClassifier
import optuna
# from tabpfn import TabPFNClassifier

C:\Users\Sebastian\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Data

In [2]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            elo = pd.read_csv(join(data_path, 'elo.csv'))
            elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
            elo = elo.drop(['CoachName'], axis=1)
            def add_elo_column(df):
                df = df.copy()
                df = pd.merge(
                        df,
                        elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                        left_on=['Season', 'DayNum', 'T1_TeamID'],
                        right_on=['Season', 'DayNum', 'TeamID'],
                        how='left'
                    )
                df = df.drop(['TeamID'], axis=1)
                return df
            df_train = add_elo_column(df_train)
            df_test = add_elo_column(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

## How to get data with women - men separation

### Set correct parameters
This the place where you change parameters

In [4]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
 'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
 'T1_FTM',
 'T1_FTA',
 'T1_OR',
 'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
 'T1_Blk',
 'T1_PF',
                      
 'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
 'T1_opponent_FTM',
 'T1_opponent_FTA',
 'T1_opponent_OR',
 'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
 'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
 'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
 'T2_FTM',
 'T2_FTA',
 'T2_OR',
 'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
 'T2_Blk',
 'T2_PF',
                      
 'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
 'T2_opponent_FTM',
 'T2_opponent_FTA',
 'T2_opponent_OR',
 'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
 'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
 'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
 'T1_FTM',
 'T1_FTA',
 'T1_OR',
 'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
 'T1_Blk',
 'T1_PF',
                      
 'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
 'T1_opponent_FTM',
 'T1_opponent_FTA',
 'T1_opponent_OR',
 'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
 'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
 'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
 'T2_FTM',
 'T2_FTA',
 'T2_OR',
 'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
 'T2_Blk',
 'T2_PF',
                      
 'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
 'T2_opponent_FTM',
 'T2_opponent_FTA',
 'T2_opponent_OR',
 'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
 'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)] # for which years will you generate
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 14 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0

### Get arrays of data frames with women-men split

In [5]:
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)

### Get arrays of data frames with women-men split with the columns as specified before + get x and y train/test

In [6]:
x_train_women_list = []
x_test_women_list = []

x_train_men_list = []
x_test_men_list = []

y_train_women_list = []
y_test_women_list = []

y_train_men_list = []
y_test_men_list = []

for i in range(len(year_range)):
    
    # HERE COLUMNS ARE SELECTED
    # YOU CAN ALSO MODIFY THE COLUMNS IN THE FOR LOOP
    # THE CODE AFTER THIS PART CREATES x AND y FORM THE DATA FRAMES
    df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
    df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
    df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
    df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
    
    # BELOW x AND y test/train ARE CREATED FROM THE DATA FRAMES ABOVE
    x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
    x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
    x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
    x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

    x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
    x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
    x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
    x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
    
    x_train_women_list.append(x_train_women)
    x_test_women_list.append(x_test_women)
    x_train_men_list.append(x_train_men)
    x_test_men_list.append(x_test_men)
    
    y_train_women_list.append(y_train_women)
    y_test_women_list.append(y_test_women)
    y_train_men_list.append(y_train_men)
    y_test_men_list.append(y_test_men)

### What we have
We have arrays of data frames.
Each array represents

In [7]:
df_train_women_list[0] # Each element of the array is a data frame for the corresponding year from year_range

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_Score_mean,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Ast,T1_TO,T1_Stl,T1_Blk,T1_PF,T1_opponent_Score,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_FTM,T1_opponent_FTA,T1_opponent_OR,T1_opponent_DR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_Blk,T1_opponent_PF,T1_PointDiff,T2_Score_mean,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,T2_opponent_Score,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_FTM,T2_opponent_FTA,T2_opponent_OR,T2_opponent_DR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_Blk,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO
0,2010,138,3124,69,3201,55,0,62.250000,21.250000,54.000000,2.750000,10.750000,17.000000,23.750000,9.750000,28.750000,11.750000,15.0,6.00,5.000000,15.250000,63.500000,23.000000,57.750000,5.250000,16.500000,12.250000,16.750000,9.500000,26.500000,10.750000,14.750000,7.250000,3.500000,18.500000,-1.250000,78.000000,27.500000,69.50,10.750000,27.750000,12.250000,16.750000,16.000000,27.250000,15.500000,13.250000,10.75,3.250000,18.000000,59.000000,24.00,57.500000,4.250000,13.750000,6.750000,16.000000,12.0,28.500000,14.00,20.750000,6.250000,1.750000,13.750000,19.000000,0.500000,0.75,4,13,-9,2047.364734
1,2010,138,3173,67,3395,66,0,73.500000,26.500000,57.000000,7.000000,19.500000,13.500000,17.500000,11.500000,29.500000,14.500000,15.5,3.00,3.500000,21.000000,66.500000,21.500000,62.500000,4.000000,12.500000,19.500000,27.000000,16.000000,20.500000,9.500000,11.500000,7.000000,4.500000,18.500000,7.000000,61.000000,18.000000,55.00,6.000000,20.500000,19.000000,23.500000,8.000000,21.500000,11.500000,13.000000,6.50,0.500000,18.000000,69.500000,24.50,50.500000,6.000000,15.500000,14.500000,19.500000,8.0,32.000000,20.00,18.000000,7.000000,2.500000,19.500000,-8.500000,0.500000,0.00,8,9,-1,1803.100984
2,2010,138,3181,72,3214,37,1,67.666667,24.333333,57.000000,6.000000,12.666667,13.000000,21.666667,13.666667,20.333333,12.000000,17.0,13.00,5.333333,17.666667,59.666667,19.666667,51.666667,5.666667,14.666667,14.666667,18.000000,12.333333,20.666667,9.666667,22.666667,6.333333,1.666667,19.333333,8.000000,58.750000,22.000000,54.25,3.500000,15.000000,11.250000,15.500000,11.000000,23.750000,11.250000,12.000000,7.50,3.750000,14.750000,42.000000,15.75,46.500000,1.250000,7.750000,9.250000,17.500000,11.0,23.250000,4.75,19.250000,3.750000,1.750000,15.000000,16.750000,1.000000,1.00,2,15,-13,2222.732883
3,2010,138,3199,75,3256,61,1,60.000000,21.000000,76.000000,4.000000,28.000000,14.000000,18.000000,28.000000,16.000000,7.000000,9.0,10.00,1.000000,22.000000,67.000000,21.000000,43.000000,8.000000,20.000000,17.000000,20.000000,5.000000,29.000000,14.000000,18.000000,6.000000,7.000000,19.000000,-7.000000,71.600000,27.800000,62.80,4.000000,12.200000,12.000000,19.000000,14.000000,28.000000,14.400000,14.400000,8.00,3.000000,13.400000,67.600000,25.80,67.200000,5.600000,20.600000,10.400000,13.600000,14.8,24.600000,14.20,13.600000,7.000000,3.800000,16.600000,4.000000,0.000000,0.80,3,14,-11,2065.191353
4,2010,138,3207,62,3265,42,0,56.000000,18.000000,68.000000,4.000000,23.000000,16.000000,22.000000,18.000000,26.000000,11.000000,16.0,11.00,2.000000,24.000000,63.000000,16.000000,55.000000,3.000000,21.000000,28.000000,36.000000,16.000000,34.000000,9.000000,22.000000,5.000000,5.000000,21.000000,-7.000000,64.000000,22.333333,54.00,4.666667,15.666667,14.666667,20.333333,8.333333,27.000000,11.333333,8.666667,9.00,5.000000,9.333333,44.666667,18.00,59.333333,4.000000,21.666667,4.666667,6.333333,15.0,25.000000,8.00,15.000000,6.333333,3.000000,18.666667,19.333333,0.000000,1.00,5,12,-7,1901.415647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [8]:
df_test_men_list[0]

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_Score_mean,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Ast,T1_TO,T1_Stl,T1_Blk,T1_PF,T1_opponent_Score,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_FTM,T1_opponent_FTA,T1_opponent_OR,T1_opponent_DR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_Blk,T1_opponent_PF,T1_PointDiff,T2_Score_mean,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,T2_opponent_Score,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_FTM,T2_opponent_FTA,T2_opponent_OR,T2_opponent_DR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_Blk,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO,CoachELO
0,2011,134,1155,70,1412,52,0,71.250000,25.250000,58.750000,7.75,19.250000,13.00,18.250000,13.25,24.250000,14.5,14.500000,9.750000,4.750000,21.25,67.250000,21.750000,55.75,6.75,19.250000,17.000000,23.0,14.00,21.75,13.250000,16.500000,7.500000,4.500000,17.000000,4.000000,67.666667,25.000000,60.666667,5.666667,24.333333,12.000000,21.000000,15.00,27.000000,12.333333,9.333333,4.000000,2.666667,20.666667,63.000000,19.000000,48.000000,5.333333,16.666667,19.666667,29.000000,7.666667,26.666667,7.666667,10.000000,4.666667,3.000000,17.666667,4.666667,0.500000,0.666667,12,12,0,1874.433076,1883.148122
1,2011,134,1421,81,1114,77,0,61.000000,23.500000,58.000000,5.00,16.500000,9.00,13.500000,14.50,22.000000,12.5,10.500000,8.500000,7.000000,18.00,46.000000,16.500000,47.00,4.50,16.000000,8.500000,14.0,12.50,22.00,8.500000,17.500000,6.000000,4.500000,16.500000,15.000000,67.500000,21.750000,49.750000,4.000000,13.250000,20.000000,25.000000,7.50,23.750000,10.500000,9.500000,6.250000,3.750000,22.250000,59.750000,19.250000,49.500000,4.250000,13.250000,17.000000,25.500000,11.750000,24.250000,9.250000,14.500000,3.750000,2.000000,21.250000,7.750000,1.000000,1.000000,16,16,0,1418.690794,1579.322844
2,2011,135,1427,70,1106,61,0,77.800000,25.400000,52.800000,9.00,19.000000,18.00,23.800000,10.20,25.200000,12.8,12.200000,6.200000,2.400000,19.60,71.800000,25.200000,63.00,7.80,26.400000,13.600000,22.0,16.40,21.20,12.800000,10.000000,5.400000,3.600000,18.600000,6.000000,71.000000,21.800000,51.600000,5.000000,11.600000,22.400000,33.400000,13.40,26.200000,11.000000,12.200000,6.800000,4.200000,21.200000,60.400000,19.200000,49.000000,5.600000,15.400000,16.400000,26.000000,9.400000,21.200000,10.600000,14.200000,5.800000,3.800000,24.000000,10.600000,1.000000,0.800000,16,16,0,1478.699359,1589.430063
3,2011,135,1433,59,1425,46,0,68.666667,21.333333,49.333333,8.00,20.666667,18.00,27.333333,9.00,19.666667,13.0,11.666667,6.666667,0.666667,20.00,64.333333,24.333333,51.00,5.00,16.333333,10.666667,20.0,12.00,23.00,10.333333,15.333333,6.333333,1.666667,20.333333,4.333333,67.750000,24.750000,54.750000,6.500000,18.250000,11.750000,17.000000,11.00,22.500000,11.750000,11.500000,5.750000,2.500000,20.000000,67.000000,24.250000,53.750000,4.000000,13.000000,14.500000,19.250000,8.750000,22.500000,10.500000,9.750000,5.250000,3.750000,15.250000,0.750000,0.666667,0.500000,11,11,0,1718.948974,1786.986088
4,2011,136,1139,60,1330,58,0,67.500000,23.000000,51.500000,8.50,23.000000,13.00,18.500000,10.50,23.500000,10.5,11.000000,6.500000,2.500000,21.50,56.000000,17.000000,46.50,3.50,16.000000,18.500000,24.0,9.50,21.50,8.000000,12.000000,7.500000,1.000000,17.500000,11.500000,68.666667,25.000000,46.000000,3.333333,10.000000,15.333333,25.000000,12.00,24.000000,11.666667,12.000000,5.333333,2.666667,12.666667,61.333333,22.333333,54.333333,9.000000,23.666667,7.666667,13.000000,10.333333,14.666667,15.666667,9.000000,5.333333,3.000000,19.666667,7.333333,1.000000,1.000000,8,9,-1,1909.710790,1961.059471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

In [9]:
x_test_men_list

[array([[7.12500000e+01, 2.52500000e+01, 5.87500000e+01, ...,
         0.00000000e+00, 1.87443308e+03, 1.88314812e+03],
        [6.10000000e+01, 2.35000000e+01, 5.80000000e+01, ...,
         0.00000000e+00, 1.41869079e+03, 1.57932284e+03],
        [7.78000000e+01, 2.54000000e+01, 5.28000000e+01, ...,
         0.00000000e+00, 1.47869936e+03, 1.58943006e+03],
        ...,
        [6.86666667e+01, 2.13333333e+01, 4.93333333e+01, ...,
         3.00000000e+00, 1.84036384e+03, 1.90616480e+03],
        [7.02500000e+01, 2.35000000e+01, 5.35000000e+01, ...,
         1.00000000e+00, 2.07715857e+03, 2.19368030e+03],
        [6.75000000e+01, 2.30000000e+01, 5.15000000e+01, ...,
         5.00000000e+00, 1.99004040e+03, 2.04587013e+03]]),
 array([[6.55000000e+01, 2.45000000e+01, 5.30000000e+01, ...,
         0.00000000e+00, 1.91065492e+03, 1.98274215e+03],
        [6.90000000e+01, 2.17500000e+01, 5.15000000e+01, ...,
         0.00000000e+00, 1.53069256e+03, 1.63652279e+03],
        [6.90000000e+01, 

In [10]:
y_test_men_list

[array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0.

## If you want train data for all seasons back and test data for just one last specified season you can use the code below

In [5]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2024]
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 150 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.9, 1.1] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0

###############################################################

df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)

###############################################################
x_train_women_list = []
x_test_women_list = []

x_train_men_list = []
x_test_men_list = []

y_train_women_list = []
y_test_women_list = []

y_train_men_list = []
y_test_men_list = []
    
df_train_women_list = df_train_women_list[0][columns_to_include_women]
df_test_women_list = df_test_women_list[0][columns_to_include_women]
df_train_men_list = df_train_men_list[0][columns_to_include_men]
df_test_men_list = df_test_men_list[0][columns_to_include_men]
    
x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

In [6]:
df_train_women_list

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_OR,T1_Ast,T1_TO,T1_Stl,T1_PF,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_OR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_PF,T1_PointDiff,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_OR,T2_Ast,T2_TO,T2_Stl,T2_PF,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_OR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO
0,2010,138,3124,69,3201,55,0,26.093750,56.343750,2.468750,8.656250,12.187500,14.687500,16.875000,6.968750,14.125000,21.125000,62.625000,5.531250,19.937500,12.687500,9.500000,15.718750,6.656250,19.250000,15.250000,25.848485,61.575758,8.818182,24.848485,13.878788,14.636364,15.636364,10.393939,16.393939,22.939394,56.696970,4.151515,13.848485,12.242424,11.848485,19.878788,6.757576,16.242424,12.878788,0.500000,0.750000,4,13,-9,2047.364734
1,2010,138,3173,67,3395,66,0,25.884615,61.538462,5.461538,17.461538,14.730769,14.461538,16.961538,7.730769,16.576923,21.576923,58.692308,5.423077,16.846154,12.076923,11.653846,18.346154,7.615385,18.230769,11.269231,25.833333,62.266667,6.866667,19.200000,13.500000,15.600000,16.033333,9.833333,16.233333,21.833333,59.700000,5.733333,20.166667,14.866667,13.466667,20.666667,8.466667,16.566667,12.000000,0.500000,0.333333,8,9,-1,1803.100984
2,2010,138,3181,72,3214,37,1,26.843750,63.750000,4.562500,14.468750,17.718750,14.250000,17.906250,13.375000,17.687500,18.843750,53.593750,5.312500,16.593750,12.500000,11.375000,23.218750,7.531250,17.875000,16.531250,22.200000,60.100000,4.900000,16.100000,14.966667,11.600000,15.533333,10.100000,18.833333,19.000000,49.933333,2.866667,10.333333,12.633333,8.733333,22.133333,6.633333,16.700000,7.700000,1.000000,1.000000,2,15,-13,2222.732883
3,2010,138,3199,75,3256,61,1,26.400000,59.200000,6.100000,16.100000,14.400000,15.333333,19.033333,8.833333,17.133333,21.333333,59.566667,5.300000,18.433333,13.400000,12.233333,19.200000,8.200000,18.933333,14.366667,27.161290,62.516129,3.935484,12.451613,14.806452,14.516129,17.096774,7.096774,16.032258,23.903226,64.064516,4.806452,17.387097,13.709677,10.741935,17.451613,8.193548,19.677419,9.935484,0.000000,0.800000,3,14,-11,2065.191353
4,2010,138,3207,62,3265,42,0,23.833333,60.266667,6.333333,20.066667,15.766667,15.866667,16.300000,13.366667,18.033333,19.733333,48.866667,5.400000,18.833333,12.100000,13.266667,24.600000,7.933333,16.966667,9.666667,23.424242,55.696970,6.545455,19.030303,10.515152,14.454545,13.484848,7.878788,13.969697,20.575758,58.939394,6.333333,21.363636,14.333333,10.727273,15.727273,7.151515,16.818182,10.272727,0.500000,1.000000,5,12,-7,1901.415647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1649,2023,147,3268,75,3376,86,-1,28.741935,65.129032,6.838710,19.129032,9.967742,16.677419,13.193548,10.322581,16.870968,25.290323,61.612903,7.806452,23.225806,10.838710,15.935484,18.483871,6.322581,17.161290,10.096774,30.718750,65.312500,4.406250,14.000000,16.000000,16.781250,12.437500,6.718750,15.031250,18.718750,59.593750,3.968750,15.562500,6.562500,8.031250,13.000000,5.656250,19.718750,30.343750,0.500000,1.000000,2,1,1,2240.902170
1650,2023,147,3326,74,3439,84,0,29.968750,64.531250,7.468750,21.937500,9.062500,17.468750,13.500000,11.437500,17.468750,25.531250,59.937500,5.937500,19.687500,8.625000,14.343750,18.906250,5.906250,17.000000,12.500000,25.967742,57.451613,7.741935,22.193548,8.967742,15.225806,12.645161,5.161290,15.354839,21.516129,56.548387,4.354839,15.129032,7.064516,9.129032,13.354839,5.935484,19.419355,15.451613,0.666667,1.000000,3,1,2,2167.546827
1651,2023,151,3376,73,3234,77,0,30.718750,65.312500,4.406250,14.000000,16.000000,16.781250,12.437500,6.718750,15.031250,18.718750,59.593750,3.968750,15.56250

In [7]:
x_train_women

array([[ 2.60937500e+01,  5.63437500e+01,  2.46875000e+00, ...,
         1.30000000e+01, -9.00000000e+00,  2.04736473e+03],
       [ 2.58846154e+01,  6.15384615e+01,  5.46153846e+00, ...,
         9.00000000e+00, -1.00000000e+00,  1.80310098e+03],
       [ 2.68437500e+01,  6.37500000e+01,  4.56250000e+00, ...,
         1.50000000e+01, -1.30000000e+01,  2.22273288e+03],
       ...,
       [ 3.07187500e+01,  6.53125000e+01,  4.40625000e+00, ...,
         2.00000000e+00, -1.00000000e+00,  2.50062656e+03],
       [ 2.59677419e+01,  5.74516129e+01,  7.74193548e+00, ...,
         3.00000000e+00, -2.00000000e+00,  2.21498932e+03],
       [ 3.15625000e+01,  6.20312500e+01,  8.90625000e+00, ...,
         3.00000000e+00, -1.00000000e+00,  2.26234638e+03]])

In [8]:
y_train_women

array([1., 1., 1., ..., 0., 0., 0.])

In [9]:
df_test_women_list

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_OR,T1_Ast,T1_TO,T1_Stl,T1_PF,T1_opponent_FGM,T1_opponent_FGA,T1_opponent_FGM3,T1_opponent_FGA3,T1_opponent_OR,T1_opponent_Ast,T1_opponent_TO,T1_opponent_Stl,T1_opponent_PF,T1_PointDiff,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_OR,T2_Ast,T2_TO,T2_Stl,T2_PF,T2_opponent_FGM,T2_opponent_FGA,T2_opponent_FGM3,T2_opponent_FGA3,T2_opponent_OR,T2_opponent_Ast,T2_opponent_TO,T2_opponent_Stl,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO
0,2024,135,3342,49,3357,42,0,21.258065,53.000000,5.419355,17.838710,7.483871,12.354839,16.516129,6.193548,13.419355,22.774194,56.870968,5.677419,18.870968,8.806452,11.677419,14.129032,9.645161,14.935484,-3.322581,24.258065,59.645161,5.483871,19.096774,10.258065,11.580645,12.451613,9.064516,16.129032,21.483871,53.903226,4.741935,16.225806,9.193548,11.419355,17.612903,5.193548,17.258065,8.645161,1.000000,1.000000,16,16,0,1260.494693
1,2024,135,3435,72,3162,68,0,25.064516,61.322581,6.548387,20.225806,11.387097,14.709677,14.096774,10.129032,16.290323,22.838710,56.935484,5.870968,20.258065,9.064516,12.032258,16.677419,7.064516,16.741935,6.967742,28.103448,61.965517,8.034483,24.482759,12.689655,16.793103,14.448276,7.413793,17.241379,23.344828,58.551724,5.413793,17.137931,9.068966,13.172414,14.275862,5.896552,16.068966,12.758621,0.000000,0.666667,12,12,0,1843.736914
2,2024,136,3112,69,3120,59,0,26.437500,60.125000,4.312500,13.718750,8.375000,14.562500,13.875000,11.562500,18.687500,23.718750,55.875000,6.218750,19.500000,9.562500,13.562500,18.812500,6.968750,16.406250,4.187500,24.709677,61.032258,3.838710,13.225806,10.193548,12.741935,14.774194,10.354839,18.258065,20.322581,53.903226,3.903226,14.000000,8.903226,8.935484,19.903226,7.354839,18.419355,8.129032,0.500000,0.500000,11,11,0,1960.469204
3,2024,136,3221,72,3404,45,0,23.161290,58.354839,5.709677,17.451613,9.838710,14.290323,12.096774,4.935484,15.903226,20.548387,56.096774,4.677419,17.419355,7.774194,10.806452,11.612903,6.096774,15.580645,5.290323,22.612903,51.387097,6.096774,16.516129,6.580645,12.000000,15.903226,8.000000,13.419355,23.451613,58.129032,7.161290,22.290323,9.903226,15.064516,13.645161,8.193548,15.483871,0.000000,1.000000,0.666667,16,16,0,1496.036285
4,2024,137,3104,82,3199,74,0,26.250000,60.343750,7.406250,21.218750,10.843750,13.031250,14.343750,8.625000,15.687500,23.062500,59.031250,4.750000,16.343750,10.312500,12.250000,16.531250,7.375000,18.156250,11.718750,28.515152,68.848485,7.151515,22.787879,9.272727,12.848485,11.363636,7.272727,14.363636,27.666667,70.666667,6.424242,21.121212,11.696970,14.484848,14.393939,5.333333,17.484848,7.515152,0.000000,0.666667,8,9,-1,1976.762398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,2024,147,3425,73,3163,80,1,26.516129,62.290323,7.548387,21.225806,10.451613,14.806452,11.806452,7.935484,16.129032,22.548387,58.129032,5.451613,17.774194,7.612903,13.354839,14.548387,6.322581,18.483871,12.516129,30.545455,61.636364,7.090909,19.787879,8.121212,19.303030,12.363636,9.818182,14.363636,21.090909,58.212121,6.121212,21.575758,7.484848,11.757576,15.969697,5.212121,16.303030,22.909091,1.000000,1.000000,1,3,-2,2156.553539
130,2024,147,3261,87,3234,94,-1,31.212121,66.818182,4.242424,13.212121,14.969697,16.545455,15.212121,10.787879,16.090909,22.818182,62.121212,6.272727,21.181818,8.787879,10.818182,18.939394,8.000000,22.151515,24.151515,33.121212,65.818182,11.303030,29.606061,9.121212,21.878788,13.424242,7.545455,14.696970,26.454545,65.969697,7.848485,25.272727,8.818182,14.242424,14.424242,7.060606,19.272727,20.939394,0.666667,1.000000,3,1,2,2242.701954
131,2024,151,3163,69,3234,71,0,30.545455,61.636364,7.090909,19.787879,8.121212,19.303030,12.363636,9.818182,14.363636,21.090909,58.212121,6.121212,21.575758,7.484848,11.75757

Here the train data is from "start_season = 2003" and the test data is from "year_range = [2024]".